# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prarthanamahesh21-hub/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### My lane as an ML task

**Task type: Ranking**

I will frame my lane as a **ranking problem**. The goal is to rank content pages by how strongly they should be considered for a possible content refresh.

The model would use page-level signals such as search performance, engagement, content age, and other available performance indicators to produce a priority ranking. The output would help an editor or SEO reviewer decide which pages to investigate first.

This is a ranking problem rather than a simple classification problem because the goal is not to automatically label a page as "needs refresh" or "does not need refresh." Instead, the goal is to determine **which pages should receive attention first**.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Target or proxy

The target will be a **content-refresh priority score** that represents how strongly a page should be considered for review.

Because the starter dataset does not provide a direct label saying whether a page actually needs a content refresh, I will use observable page-level performance and content-age signals as a **proxy** for refresh priority.

The proxy should capture pages that show signs such as declining or weak search performance, high impressions with relatively low clicks, lower engagement, or older content that may benefit from review.

The model's output will therefore be a ranking score rather than a definitive "refresh" label. Higher-scoring pages would be placed earlier in the review queue.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Success metric

The primary success metric will be **NDCG@K (Normalized Discounted Cumulative Gain)**.

NDCG@K is appropriate because this is a ranking problem where the most important pages should appear near the top of the review queue. It measures how well the ranking places higher-priority pages ahead of lower-priority pages.

Using a value of K focuses evaluation on the first set of pages that an editor or SEO reviewer is most likely to investigate. A higher NDCG@K means that the ranking is doing a better job of putting the most useful pages near the top.

The practical goal is therefore not only to produce a ranking, but to make the **top of the ranking useful for real content-review decisions**.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://huggingface.co/datasets/FlyRank/internship-starter/resolve/main/content_refresh_anonymized.csv"

df = pd.read_csv(url)

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_pct,health_score,needs_indexing,is_quick_win,needs_ctr_fix,needs_engagement_fix,ai_opportunity,is_underperformer,is_declining,is_initial_refresh_candidate
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,-41.4,50,False,False,False,False,False,False,True,True
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,-57.7,40,False,True,False,False,False,False,True,True
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,-60.9,40,False,False,False,False,False,False,True,False
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,-13.8,60,False,False,False,True,False,False,False,True
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,-34.7,40,False,False,False,True,True,False,True,False


In [15]:
df.shape


(30000, 53)

In [16]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct',
 'health_score',
 'needs_indexing',
 'is_quick_win',
 'needs_ctr_fix',
 'needs_engagement_fix',
 'ai_opportunity',
 'is_underperformer',
 'is_declining',
 'is_initial_refresh_candidate']

In [17]:
page_df = df[
    [
        "content_id",
        "content_type",
        "content_age_days",
        "impressions_90d",
        "clicks_90d",
        "pageviews_90d",
        "sessions_90d",
        "engagement_rate",
        "avg_position",
        "trend_direction",
        "trend_pct"
    ]
]

page_df.head(10)

,content_id,content_type,content_age_days,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,engagement_rate,avg_position,trend_direction,trend_pct
0,content_304f48230142,keyword article,187,3803,29,22,17,5.88,10.6,down,-41.4
1,content_a1fb4e703a9e,keyword article,445,15320,7,10,9,0.00,20.3,down,-57.7
2,content_9aa793d4d895,keyword article,141,12581,11,14,11,0.00,36.5,down,-60.9
3,content_331d6c4de07b,keyword article,463,11751,58,87,78,1.28,6.2,stable,-13.8
4,content_d99b7a2d90ca,keyword article,263,19140,24,177,145,0.00,44.0,down,-34.7
5,content_d4084a4bc775,keyword article,147,3970,1,4,5,0.00,8.5,down,-38.9
6,content_9a34b442b552,keyword article,90,20,0,1,1,0.00,7.0,down,-92.3
7,content_a63219c6e95a,keyword article,445,1724,1,28,28,3.57,21.2,stable,0.6
8,content_5e6c160719bc,keyword article,90,32574,29,128,68,5.88,46.0,down,-58.8
9,content_c27558df2b0c,keyword article,257,1240,2,4,3,0.00,4.9,down,-29.2


### The unit of analysis

**One row = one content page.**

The unit of analysis is the individual content page identified by `content_id`. Each row contains page-level characteristics and performance signals, including content age, search impressions, clicks, pageviews, sessions, engagement rate, average search position, and recent performance trends.

This unit of analysis matches the task because the decision we want to support is **which individual content pages should be reviewed first for a possible refresh**.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
### Why ML beats a fixed rule here



A fixed rule would be too limited because content-refresh priority depends on multiple signals rather than a single condition. For example, simply prioritizing pages older than a certain number of days could incorrectly flag old pages that are still performing well, while missing newer pages whose search performance or engagement is declining.

An ML-based ranking approach can learn how different signals work together, such as content age, impressions, clicks, engagement, search position, and recent performance trends. This allows the system to identify more useful patterns than a manually chosen threshold.

ML is therefore useful for producing a more flexible priority ranking that can help an editor or SEO reviewer decide which pages to investigate first. The model would support the review decision rather than automatically deciding that a page must be refreshed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-check

* **ML task type:** Ranking
* **Target/proxy:** Content-refresh priority score
* **Success metric:** NDCG@K
* **Unit of analysis:** One row = one content page
* **Real dataframe:** Page-level content and performance signals are displayed in the notebook
* **Why ML instead of a fixed rule:** Multiple signals can be combined to produce a more useful ranking than a single manually chosen threshold
* **Action supported:** The ranking helps an editor or SEO reviewer decide which content pages to investigate first for a possible refresh

The framing is intentionally a prioritization aid rather than an automatic decision about whether a page must be refreshed.
